# CE only

## 1. Imports

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tqdm import tqdm

from archive.z_cnn_experiments.cnn import StandardCNN
from src.dataset._balance_class_subset import BalancedClassSubset
from src.dataset.tiny_imagenet_dataset_450train_100val import TinyImageNetLoader, show_image, get_tiny_imagenet_class_names

print("PyTorch:", torch.__version__)


## 2. Configuration

In [ ]:
SEED = 42

NUM_CLASSES = 200
SELECTED_CLASSES = list(range(NUM_CLASSES))
TRAIN_IMAGES_PER_CLASS = 500
VAL_IMAGES_PER_CLASS = 50

BATCH_SIZE = 32
NUM_WORKERS = 0
EPOCHS = 90

MAX_LR = 0.06
SGD_WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.10

FEATURE_DROPOUT = 0.40
BLOCK3_DROPOUT = 0.15
BLOCK4_DROPOUT = 0.25

EARLY_STOPPING_PATIENCE = 30
SAVE_PATH = f"ce_only_{NUM_CLASSES}_2.pth"
TINY_ROOT = "../../data/tiny_imagenet"



## 3. Reproducibility and Device

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available(): DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available(): DEVICE = torch.device("mps")
else: DEVICE = torch.device("cpu")

print("Device:", DEVICE)


## 6. Load Tiny ImageNet

In [ ]:
train_dataset, val_dataset, _, _ = TinyImageNetLoader(seed=SEED, root=Path(TINY_ROOT)).load()
train_eval_dataset, _, _, _ = TinyImageNetLoader(seed=SEED, root=Path(TINY_ROOT)).load()

print("Full train dataset:", len(train_dataset))
print("Full validation dataset:", len(val_dataset))
print("Full clean train-eval dataset:", len(train_eval_dataset))


## 7. Select the Same 20 Classes

In [ ]:
train_subset = BalancedClassSubset(train_dataset, SELECTED_CLASSES, TRAIN_IMAGES_PER_CLASS, seed=SEED)
train_eval_subset = BalancedClassSubset(train_eval_dataset, SELECTED_CLASSES, TRAIN_IMAGES_PER_CLASS, seed=SEED)
val_subset = BalancedClassSubset(val_dataset, SELECTED_CLASSES, VAL_IMAGES_PER_CLASS, seed=SEED + 1)

FULL_CLASS_NAMES = get_tiny_imagenet_class_names(train_dataset, TINY_ROOT)
CLASS_NAMES = [FULL_CLASS_NAMES[class_id] for class_id in SELECTED_CLASSES]

print("Same selected training samples:", train_subset.samples == train_eval_subset.samples)
print("Selected classes:", len(SELECTED_CLASSES))
print("Training images:", len(train_subset))
print("Clean train-eval images:", len(train_eval_subset))
print("Validation images:", len(val_subset))


## 8. DataLoaders

In [ ]:
generator = torch.Generator()
generator.manual_seed(SEED)
persistent_workers = NUM_WORKERS > 0

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=NUM_WORKERS, persistent_workers=persistent_workers)
train_eval_loader = DataLoader(train_eval_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=persistent_workers)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, persistent_workers=persistent_workers)

print("Training batches:", len(train_loader))
print("Clean train-eval batches:", len(train_eval_loader))
print("Validation batches:", len(val_loader))


## Show Images

In [ ]:
NUM_SHOW = 10

train_iter = iter(train_loader)
val_iter = iter(val_loader)

In [ ]:
images, labels = next(train_iter)
show_image(images, labels, NUM_SHOW, CLASS_NAMES, name='Train Images')

images, labels = next(train_iter)
show_image(images, labels, NUM_SHOW, CLASS_NAMES, name='Validation Images')

## 9. Create Model

In [ ]:
model = StandardCNN(num_classes=NUM_CLASSES, feature_dropout=FEATURE_DROPOUT, block3_dropout=BLOCK3_DROPOUT, block4_dropout=BLOCK4_DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable parameters (M): {trainable_params / 1e6:.3f}M")

x = torch.randn(2, 3, 64, 64, device=DEVICE)

model.eval()
with torch.no_grad():
    r1 = model.level1(x)
    r2 = model.level2(r1)
    r3 = model.level3(r2)
    r4 = model.level4(r3)
    features = torch.flatten(model.global_pool(r4), 1)
    logits = model(x)

print("Input:", x.shape)
print("r1:", r1.shape)
print("r2:", r2.shape)
print("r3:", r3.shape)
print("r4:", r4.shape)
print("Features:", features.shape)
print("Logits:", logits.shape)

model.train()


## 10. Loss, Optimizer and OneCycleLR

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.SGD(model.parameters(), lr=MAX_LR, momentum=0.9, weight_decay=SGD_WEIGHT_DECAY, nesterov=True)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.10, anneal_strategy="cos", div_factor=10.0,
    final_div_factor=10.0
)

print("Loss: CrossEntropyLoss")
print("Label smoothing:", LABEL_SMOOTHING)
print("Optimizer: SGD + Nesterov")
print("Weight decay:", SGD_WEIGHT_DECAY)
print("OneCycle max LR:", MAX_LR)
print("Initial scheduled LR:", optimizer.param_groups[0]["lr"])
print("Epochs:", EPOCHS)
print("Steps per epoch:", len(train_loader))


## 11. Training Pass

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion, scheduler, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        correct += (predictions == labels).sum().item()
        total += batch_size

    return {"loss": running_loss / total, "accuracy": 100.0 * correct / total}


## 12. Evaluation Pass

In [ ]:
@torch.inference_mode()
def evaluate(model, data_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predictions = []

    for images, labels in data_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        correct += (predictions == labels).sum().item()
        total += batch_size
        all_labels.append(labels.cpu())
        all_predictions.append(predictions.cpu())

    return {"loss": running_loss / total, "accuracy": 100.0 * correct / total, "labels": torch.cat(all_labels), "predictions": torch.cat(all_predictions)}


## 13. Train Model

In [ ]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "train_eval_loss": [],
    "train_eval_accuracy": [],
    "train_val_gap": [],
    "clean_train_eval_val_gap": [],
    "learning_rate": []
}

best_val_accuracy = 0.0
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in tqdm(range(1, EPOCHS + 1)):
    # PHASE 1: actual training pass
    train_result = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, DEVICE)

    # PHASE 2: validation pass
    val_result = evaluate(model, val_loader, criterion, DEVICE)

    # ADDITIONAL DIAGNOSTIC: clean evaluation of the same training samples
    train_eval_result = evaluate(model, train_eval_loader, criterion, DEVICE)

    current_lr = optimizer.param_groups[0]["lr"]
    train_val_gap = train_result["accuracy"] - val_result["accuracy"]
    clean_train_eval_val_gap = train_eval_result["accuracy"] - val_result["accuracy"]

    history["train_loss"].append(train_result["loss"])
    history["train_accuracy"].append(train_result["accuracy"])
    history["val_loss"].append(val_result["loss"])
    history["val_accuracy"].append(val_result["accuracy"])
    history["train_eval_loss"].append(train_eval_result["loss"])
    history["train_eval_accuracy"].append(train_eval_result["accuracy"])
    history["train_val_gap"].append(train_val_gap)
    history["clean_train_eval_val_gap"].append(clean_train_eval_val_gap)
    history["learning_rate"].append(current_lr)

    improved = val_result["accuracy"] > best_val_accuracy or (val_result["accuracy"] == best_val_accuracy and val_result["loss"] < best_val_loss)

    if improved:
        best_val_accuracy = val_result["accuracy"]
        best_val_loss = val_result["loss"]
        best_epoch = epoch
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "train_loss": train_result["loss"],
            "train_accuracy": train_result["accuracy"],
            "val_loss": val_result["loss"],
            "val_accuracy": val_result["accuracy"],
            "train_eval_loss": train_eval_result["loss"],
            "train_eval_accuracy": train_eval_result["accuracy"],
            "train_val_gap": train_val_gap,
            "clean_train_eval_val_gap": clean_train_eval_val_gap,
            "learning_rate": current_lr,
            "num_classes": NUM_CLASSES,
            "label_smoothing": LABEL_SMOOTHING,
            "max_lr": MAX_LR,
            "weight_decay": SGD_WEIGHT_DECAY,
            "architecture": "48-96-192-384"
        }

        torch.save(checkpoint, SAVE_PATH)
        print(f"  -> Best model saved | Epoch: {epoch} | Val Acc: {val_result['accuracy']:.2f}% | Val Loss: {val_result['loss']:.4f}")
    else:
        epochs_without_improvement += 1

    print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] | LR: {current_lr:.6f}")
    print(f"Train      -> Loss: {train_result['loss']:.4f} | Acc: {train_result['accuracy']:.2f}%")
    print(f"Validation -> Loss: {val_result['loss']:.4f} | Acc: {val_result['accuracy']:.2f}%")
    print(f"Train-Val Gap: {train_val_gap:.2f} pp")
    print("Additional:")
    print(f"Clean Train Eval -> Loss: {train_eval_result['loss']:.4f} | Acc: {train_eval_result['accuracy']:.2f}%")
    print(f"Clean Train Eval-Val Gap: {clean_train_eval_val_gap:.2f} pp")
    print(f"Best Val Acc: {best_val_accuracy:.2f}% @ Epoch {best_epoch}")
    print("-" * 90)

    if EARLY_STOPPING_PATIENCE is not None and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}. No validation improvement for {EARLY_STOPPING_PATIENCE} epochs.")
        break

print("=" * 90)
print(f"Best Epoch: {best_epoch} | Best Val Acc: {best_val_accuracy:.2f}% | Best Val Loss: {best_val_loss:.6f}")
print("Best model saved to:", SAVE_PATH)


In [ ]:
# Experiment:
#   StandardCNN 48-96-192-384
#   RandAugment + CE
#
# Total training loss:
#   CE(clean)

## 14. Load Best Model and Re-Evaluate

In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

best_train_eval_result = evaluate(model, train_eval_loader, criterion, DEVICE)
best_val_result = evaluate(model, val_loader, criterion, DEVICE)

best_train_val_gap = checkpoint["train_accuracy"] - best_val_result["accuracy"]
best_clean_train_eval_val_gap = best_train_eval_result["accuracy"] - best_val_result["accuracy"]

print("STANDARD CNN 384 — CE only")
print("=" * 80)
print(f"Best Epoch: {checkpoint['epoch']}")
print(f"Train      -> Loss: {checkpoint['train_loss']:.4f} | Acc: {checkpoint['train_accuracy']:.2f}%")
print(f"Validation -> Loss: {best_val_result['loss']:.4f} | Acc: {best_val_result['accuracy']:.2f}%")
print(f"Train-Val Gap: {best_train_val_gap:.2f} pp")
print("Additional:")
print(f"Clean Train Eval -> Loss: {best_train_eval_result['loss']:.4f} | Acc: {best_train_eval_result['accuracy']:.2f}%")
print(f"Clean Train Eval-Val Gap: {best_clean_train_eval_val_gap:.2f} pp")
print("=" * 80)


## t-SNE - visualizing local class clusters

In [ ]:
# import importlib
# import new_experiment.best_models.cnn as cnn_1
#
# importlib.reload(cnn_1)

model = StandardCNN(num_classes=NUM_CLASSES, feature_dropout=FEATURE_DROPOUT, block3_dropout=BLOCK3_DROPOUT, block4_dropout=BLOCK4_DROPOUT).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable parameters (M): {trainable_params / 1e6:.3f}M")

x = torch.randn(2, 3, 64, 64, device=DEVICE)

model.eval()
with torch.no_grad():
    r1 = model.level1(x)
    r2 = model.level2(r1)
    r3 = model.level3(r2)
    r4 = model.level4(r3)
    features = torch.flatten(model.global_pool(r4), 1)
    logits = model(x)

print("Input:", x.shape)
print("r1:", r1.shape)
print("r2:", r2.shape)
print("r3:", r3.shape)
print("r4:", r4.shape)
print("Features:", features.shape)
print("Logits:", logits.shape)



In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import torch


@torch.inference_mode()
def extract_features(model, data_loader, device):
    model.eval()

    all_features = []
    all_labels = []

    for images, labels in data_loader:
        images = images.to(device)

        result = model(images, return_features=True)

        features = result["features"]

        all_features.append(features.cpu())
        all_labels.append(labels.cpu())

    features = torch.cat(all_features, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    return features, labels

val_features, val_labels = extract_features(
    model,
    val_loader,
    DEVICE
)

print("Feature shape:", val_features.shape)
print("Label shape:", val_labels.shape)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    max_iter=1500,
    random_state=42
)

features_2d = tsne.fit_transform(val_features)

print("2D feature shape:", features_2d.shape)

plt.figure(figsize=(10, 6))

for class_id in range(NUM_CLASSES):
    mask = val_labels == class_id

    plt.scatter(
        features_2d[mask, 0],
        features_2d[mask, 1],
        s=25,
        alpha=0.70,
        label=CLASS_NAMES[class_id]
    )

plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.title("StandardCNN — CE-only — 384-D Validation Features Projected to 2-D")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

# Interactive 3D t-SNE

In [ ]:
from sklearn.manifold import TSNE
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display

tsne_3d = TSNE(n_components=3, perplexity=30, learning_rate="auto", init="pca", max_iter=1500, random_state=42)
features_3d = tsne_3d.fit_transform(val_features)

print("3D feature shape:", features_3d.shape)

tsne_df = pd.DataFrame({
    "Dim1": features_3d[:, 0],
    "Dim2": features_3d[:, 1],
    "Dim3": features_3d[:, 2],
    "Class": [CLASS_NAMES[label] for label in val_labels]
})

fig = px.scatter_3d(
    tsne_df,
    x="Dim1",
    y="Dim2",
    z="Dim3",
    color="Class",
    opacity=0.75,
    title="StandardCNN — AugMix + CE + JSD — 256-D Validation Features Projected to 3-D"
)

fig.update_traces(marker=dict(size=4))

fig.update_layout(
    width=720,
    height=480,
    scene=dict(
        xaxis_title="t-SNE Dimension 1",
        yaxis_title="t-SNE Dimension 2",
        zaxis_title="t-SNE Dimension 3"
    )
)

display(HTML(fig.to_html(include_plotlyjs=True, full_html=False)))

In [ ]:
import numpy as np


def calculate_feature_separation(features, labels, num_classes):
    class_centers = []
    intra_distances = []

    # ---------------------------------------------------------
    # 1. Class centers + intra-class distances
    # ---------------------------------------------------------
    for class_id in range(num_classes):
        class_features = features[labels == class_id]

        center = class_features.mean(axis=0)
        class_centers.append(center)

        distances = np.linalg.norm(
            class_features - center,
            axis=1
        )

        intra_distances.extend(distances)

    class_centers = np.stack(class_centers)

    mean_intra_distance = np.mean(intra_distances)

    # ---------------------------------------------------------
    # 2. Inter-class centroid distances
    # ---------------------------------------------------------
    inter_distances = []

    for i in range(num_classes):
        for j in range(i + 1, num_classes):
            distance = np.linalg.norm(
                class_centers[i] - class_centers[j]
            )

            inter_distances.append(distance)

    mean_inter_distance = np.mean(inter_distances)

    # ---------------------------------------------------------
    # 3. Separation ratio
    # ---------------------------------------------------------
    separation_ratio = (
        mean_inter_distance / mean_intra_distance
    )

    return {
        "mean_intra_distance": mean_intra_distance,
        "mean_inter_distance": mean_inter_distance,
        "separation_ratio": separation_ratio
    }


baseline_separation = calculate_feature_separation(
    val_features,
    val_labels,
    NUM_CLASSES
)

print("FEATURE SEPARATION — CE-only")
print("=" * 60)
print(
    f"Mean Intra-Class Distance : "
    f"{baseline_separation['mean_intra_distance']:.6f}"
)
print(
    f"Mean Inter-Class Distance : "
    f"{baseline_separation['mean_inter_distance']:.6f}"
)
print(
    f"Inter / Intra Ratio       : "
    f"{baseline_separation['separation_ratio']:.6f}"
)

## 15. Training Pass and Validation Accuracy

In [ ]:
epochs_ran = range(1, len(history["train_accuracy"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_accuracy"], label="Train Pass Accuracy")
plt.plot(epochs_ran, history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("StandardCNN-384 — Train Pass vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 16. Training Pass and Validation Loss

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_loss"], label="Train Pass Loss")
plt.plot(epochs_ran, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("StandardCNN-384 — Train Pass vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 17. Clean Train Evaluation vs Validation

In [ ]:
epochs_ran = range(1, len(history["train_eval_accuracy"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_eval_accuracy"], label="Clean Train Eval Accuracy")
plt.plot(epochs_ran, history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("StandardCNN-384 — Clean Train Eval vs Validation Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_eval_loss"], label="Clean Train Eval Loss")
plt.plot(epochs_ran, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("StandardCNN-384 — Clean Train Eval vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 18. Generalization Gaps

In [ ]:
epochs_ran = range(1, len(history["train_val_gap"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["train_val_gap"], label="Train - Validation Gap")
plt.plot(epochs_ran, history["clean_train_eval_val_gap"], label="Clean Train Eval - Validation Gap")
plt.axhline(0)
plt.xlabel("Epoch")
plt.ylabel("Accuracy Gap (pp)")
plt.title("StandardCNN-384 — Generalization Gaps")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 19. OneCycleLR Curve

In [ ]:
epochs_ran = range(1, len(history["learning_rate"]) + 1)

plt.figure(figsize=(9, 5))
plt.plot(epochs_ran, history["learning_rate"])
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("StandardCNN-384 — OneCycleLR")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 20. Class-Wise Validation Accuracy

In [ ]:
labels = best_val_result["labels"]
predictions = best_val_result["predictions"]

classwise_rows = []

for class_id in range(NUM_CLASSES):
    mask = labels == class_id
    total = mask.sum().item()
    correct = (predictions[mask] == class_id).sum().item()
    false = total - correct
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    classwise_rows.append([class_id, CLASS_NAMES[class_id], total, correct, false, accuracy])

classwise_df = pd.DataFrame(classwise_rows, columns=["Class", "Class Name", "Total", "Correct", "False", "Accuracy (%)"])
display(classwise_df)


## 21. Confusion Matrix — Plain Numeric Table

In [ ]:
def confusion_matrix_from_tensors(labels, predictions, num_classes):
    matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    for true_label, predicted_label in zip(labels, predictions): matrix[int(true_label), int(predicted_label)] += 1
    return matrix


confusion_matrix = confusion_matrix_from_tensors(best_val_result["labels"], best_val_result["predictions"], NUM_CLASSES)
confusion_matrix_df = pd.DataFrame(confusion_matrix.numpy(), index=[f"True {i}" for i in range(NUM_CLASSES)], columns=[f"Pred {i}" for i in range(NUM_CLASSES)])

styled_confusion_matrix = confusion_matrix_df.style.set_properties(**{"font-size": "9px"}).set_table_styles([
    {"selector": "th", "props": [("font-size", "9px")]},
    {"selector": "td", "props": [("font-size", "9px")]}
])

print("Rows = True classes | Columns = Predicted classes")
display(styled_confusion_matrix)